# DISFASIA — Inteligência Artificial Pictórica (IAP)
## Atlas Topológico com Algoritmo JP + Gemma 4 31B

**Projeto:** Atlas Topológico da Disfasia — IAP (Inteligência Artificial Pictórica)  
**Autor:** João Pedro Pereira Passos (UFT — Universidade Federal do Tocantins, 2024)  
**Competição:** [Gemma 4 Good Hackathon — Kaggle / Google DeepMind](https://www.kaggle.com/competitions/gemma-4-good)  
**Repositório:** [github.com/joaopedropassostocantins/AFASIA](https://github.com/joaopedropassostocantins/AFASIA) ← branch `disfasia`

---

## Motivação: por que Disfasia?

A **disfasia** é um distúrbio específico do desenvolvimento da linguagem oral — diferentemente da afasia (adquirida por AVC), a disfasia manifesta-se na infância como dificuldade persistente na **organização e fluência da fala**. Crianças com disfasia pensam em conceitos antes de conseguir estruturá-los em palavras — elas operam no **espaço topológico pré-linguístico** que a IAP modela.

## Teoria IAP e Algoritmo JP (Passos, 2024)

A **Inteligência Artificial Pictórica (IAP)** propõe uma camada semântica *pré-linguística*: em vez de processar texto, opera diretamente sobre **estruturas topológicas do significado** — representadas como vetores 12D e analisadas com homologia persistente.

### Equação Central: $G \\approx I + F$

| Variável | Nome | Descrição |
|----------|------|-----------|
| $G$ | **Dinâmica do Conhecimento** | O estado em transformação — o conceito em movimento |
| $I$ | **Incerteza** | O espaço topológico atual, incompleto, do usuário |
| $F$ | **Flexibilidade** | A capacidade de encontrar o próximo passo semântico ótimo |

### Pipeline deste Notebook (100% Python puro)

```
38 pictogramas ARASAAC (palavra em pt-BR + categoria)
    ↓ [Gemma 4 31B via API Google AI Studio]
Vetores semânticos 12D (dimensões IAP)
    ↓ [Homologia Persistente H0 — Python puro, numpy]
Diagramas de persistência Vietoris-Rips simplificados (38 diagramas)
    ↓ [Wasserstein via scipy.optimize.linear_sum_assignment]
Matriz Wasserstein 38×38 (distância topológica)
    ↓ [sklearn.manifold.MDS + matplotlib]
Atlas Pictórico da Disfasia
```

**Por que é original?** Sistemas comuns usam cosseno (mede *direção*). O Algoritmo JP usa Wasserstein topológica (mede *forma*) — aplicado a CAA/disfasia, inédito na literatura (Passos, 2024).

In [ ]:
# =============================================================================
# CÉLULA 1 — SETUP & CONFIGURAÇÃO
# Única dependência externa necessária: google-generativeai
# Tudo mais (numpy, scipy, sklearn, matplotlib, Pillow, requests) já está no Kaggle
# =============================================================================

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-generativeai"])

import os, re, json, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from io import BytesIO
from scipy.optimize import linear_sum_assignment
from sklearn.manifold import MDS
from sklearn.decomposition import PCA
import google.generativeai as genai

try:
    from PIL import Image
    import requests
    PIL_OK = True
    print("[OK] PIL + requests disponíveis (imagens ARASAAC serão exibidas)")
except ImportError:
    PIL_OK = False
    print("[AVISO] PIL/requests indisponível — grid de imagens desativado")

# ── Kaggle Secret / Env Var ──────────────────────────────────────────────────
GEMINI_API_KEY = None
try:
    from kaggle_secrets import UserSecretsClient
    GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
    print("[OK] GEMINI_API_KEY via Kaggle Secrets")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if GEMINI_API_KEY:
        print("[OK] GEMINI_API_KEY via variável de ambiente")
    else:
        print("[AVISO] Sem GEMINI_API_KEY — vetores pré-computados (Gemma 4 31B, 09/04/2026) serão usados.")

GEMMA_API_ATIVO = bool(GEMINI_API_KEY)
if GEMMA_API_ATIVO:
    genai.configure(api_key=GEMINI_API_KEY)
    GEMMA_MODEL_NAME = "gemma-4-31b-it"
    gemma_model = genai.GenerativeModel(GEMMA_MODEL_NAME)
    print(f"[OK] Gemma 4 31B via API ({GEMMA_MODEL_NAME})")
else:
    gemma_model = None
    GEMMA_MODEL_NAME = "gemma-4-31b-it (pré-computado)"

import numpy, scipy, sklearn, matplotlib
print(f"[OK] numpy {numpy.__version__} | scipy {scipy.__version__} | sklearn {sklearn.__version__}")
print("[PRONTO] Todas as dependências carregadas.")

In [ ]:
# =============================================================================
# CÉLULA 2 — ARQUITETURA DO ALGORITMO JP
# =============================================================================

print("""
╔══════════════════════════════════════════════════════════════════════════════╗
║         ALGORITMO JP — INTELIGÊNCIA ARTIFICIAL PICTÓRICA (IAP)              ║
║         João Pedro Pereira Passos · UFT · Hackathon Gemma 4 Good 2026       ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  FASE 1 — PRÉ-CAMADA SEMÂNTICA (Gemma 4 31B, executa UMA VEZ por ícone)   ║
║    palavra em pt-BR → prompt IAP 12D → Gemma 4 31B → vetor [0..10]^12     ║
║    Ex: frustrado → {emocionalidade:9, urgencia:8, social:7, ...}           ║
║    Armazenado em cache permanente — sem custo recorrente de inferência      ║
║                                                                              ║
║  FASE 2 — INTELIGÊNCIA FLUIDA (Python puro, scipy, sklearn)                ║
║    vetor 12D → diagrama H0 (Vietoris-Rips 1D) → Wasserstein → MDS         ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  POR QUE WASSERSTEIN SUPERA COSSENO?                                        ║
║    Cosseno: mede DIREÇÃO (ângulo entre vetores)                             ║
║    Wasserstein: mede FORMA (estrutura de eventos topológicos)               ║
║                                                                              ║
║    Para vetores 12D em R¹: as 12 coordenadas são 12 pontos em uma linha.   ║
║    Os gaps entre pontos adjacentes (ordenados) formam o diagrama H0.        ║
║    Wasserstein = matching ótimo de gaps via scipy.linear_sum_assignment.    ║
║    Aplicado a pictogramas CAA/disfasia: inédito (Passos, UFT, 2024).       ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# =============================================================================
# CÉLULA 3 — 12 DIMENSÕES IAP + 38 ÍCONES DISFASIA + VETORES PRÉ-COMPUTADOS
# =============================================================================

# ── 12 Dimensões Semânticas IAP (Algoritmo JP, Passos 2024) ─────────────────
IAP_DIMENSOES = [
    "concretude",        # 0:  quão concreto/tangível (10=muito concreto, 0=abstrato)
    "emocionalidade",    # 1:  carga emocional (10=muito emocional, 0=neutro)
    "acao",              # 2:  ação/movimento (10=ação intensa, 0=estático)
    "social",            # 3:  interação social (10=altamente social, 0=isolado)
    "urgencia",          # 4:  urgência/prioridade (10=urgente, 0=rotineiro)
    "temporalidade",     # 5:  relação com tempo (10=muito temporal, 0=atemporal)
    "localizacao",       # 6:  espaço/lugar (10=espacial, 0=sem localização)
    "saude_corpo",       # 7:  saúde/corpo (10=muito corporal, 0=abstrato)
    "comunicacao",       # 8:  comunicação (10=central para CAA, 0=não comunicativo)
    "cognitivo",         # 9:  complexidade cognitiva (10=complexo, 0=simples)
    "necessidade_basica",# 10: necessidade básica (10=fundamental, 0=opcional)
    "especializado_caa", # 11: específico de CAA/AAC (10=vocabulário central CAA, 0=genérico)
]
N_DIM = len(IAP_DIMENSOES)  # 12

print("12 DIMENSÕES SEMÂNTICAS IAP (Algoritmo JP, Passos 2024):")
for i, d in enumerate(IAP_DIMENSOES):
    print(f"  {i+1:2d}. {d}")

# ── Vetores pré-computados pelo Gemma 4 31B (09/04/2026) ─────────────────────
# Servem como fallback completo offline. Formato: {id_arasaac: [d0..d11]}
PRECOMPUTED_VECTORS = {
    7196:  [4, 5, 9, 6, 9, 7, 2, 6, 8, 3, 8, 10],  # parar
   24998:  [2, 3, 7, 5, 3, 9, 1, 2, 8, 6, 4,  9],  # continuar
    4676:  [2, 2, 5, 3, 1, 9, 0, 3, 8, 4, 3,  7],  # devagar
    5306:  [2, 3, 9, 2, 8,10, 0, 2, 4, 3, 3,  8],  # rapido
   36914:  [1, 5, 2, 4, 3,10, 1, 2, 8, 7, 4,  9],  # esperar
   11752:  [2, 2, 8, 7, 3, 8, 1, 1,10, 5, 6,  9],  # repetir
    7158:  [1, 2, 3, 9, 2,10, 0, 0,10, 6, 3,  8],  # vez
   37753:  [2, 2, 3, 4, 3,10, 4, 1, 8, 7, 4,  9],  # primeiro
   32749:  [1, 1, 3, 4, 2,10, 0, 0, 8, 8, 5,  9],  # depois
   32747:  [1, 2, 4, 5, 8,10, 0, 1, 9, 6, 7, 10],  # agora
   32745:  [1, 1, 2, 3, 2,10, 1, 1, 8, 7, 3,  8],  # antes
   38278:  [1, 2, 1, 3, 2,10, 0, 0, 7, 7, 3,  8],  # amanha
   38279:  [1, 2, 1, 4, 1,10, 1, 0, 8, 8, 3,  8],  # ontem
    5431:  [2, 3, 5, 4, 3,10, 1, 1, 7, 6, 3,  8],  # inicio
    5358:  [1, 3, 2, 4, 3,10, 1, 2, 7, 5, 3,  9],  # fim
   40986:  [1, 9, 4, 7, 8, 2, 1, 5,10, 7, 8,  9],  # frustrado
   31310:  [1, 7, 1, 4, 0, 2, 0, 5, 4, 5, 3,  8],  # tranquilo
   30484:  [1,10, 3, 4, 7, 6, 2, 8, 9, 7, 7,  8],  # ansioso
    9907:  [1,10, 3, 8, 2, 3, 0, 4, 9, 4, 7, 10],  # feliz
   35545:  [1,10, 2, 6, 5, 3, 0, 6, 9, 6, 8,  9],  # triste
   30391:  [1,10, 4, 6, 7, 4, 1, 8, 9, 6, 8,  9],  # nervoso
   31408:  [1, 9, 2, 7, 2, 3, 0, 2, 6, 7, 2,  4],  # orgulhoso
    5382:  [4, 2, 2, 6, 4, 1,10, 3, 9, 3, 7, 10],  # aqui
    5375:  [3, 1, 2, 7, 3, 0,10, 1, 9, 2, 8, 10],  # ali
   30385:  [3, 2, 3, 2, 2, 1,10, 1, 6, 5, 4,  8],  # longe
   30383:  [4, 2, 3, 4, 3, 3,10, 2, 7, 5, 6,  9],  # perto
    5439:  [7, 1, 3, 2, 2, 0,10, 2, 8, 5, 7,  9],  # dentro
    5475:  [4, 2, 5, 3, 4, 1,10, 2, 7, 3, 8,  9],  # fora
   31724:  [4, 1, 3, 1, 2, 0,10, 2, 7, 2, 5,  9],  # cima
   25839:  [4, 1, 1, 1, 1, 0, 9, 3, 4, 5, 2,  7],  # baixo
    6517:  [3, 4, 8,10, 5, 3, 2, 7,10, 7, 9, 10],  # falar
    6572:  [10,3, 6, 8, 3, 5, 4, 8,10, 3, 7,  8],  # ouvir
   11697:  [1, 3, 2, 8, 4, 2, 0, 2,10, 9, 7,  8],  # entender
    8579:  [1, 3, 1, 9, 2, 4, 1, 1,10, 9, 4,  7],  # explicar
    9847:  [2, 3, 6, 9, 5, 3, 1, 2,10, 7, 8,  9],  # perguntar
    9031:  [3, 4, 8,10, 5, 6, 2, 3,10, 7, 8,  9],  # responder
   12252:  [2, 7, 7,10, 8, 3, 2, 6,10, 3,10, 10],  # ajuda
   41093:  [1, 3, 2,10, 6, 3, 0, 0,10, 7, 9, 10],  # nao entendi
}

# ── 38 Ícones ARASAAC da Disfasia ─────────────────────────────────────────────
# IDs reais. Licença: Creative Commons BY-NC-SA (https://arasaac.org)
DISFASIA_ICONS = [
    # fluencia (7) — controle do fluxo da fala
    {"id": 7196,  "palavra": "parar",       "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/7196/7196_500.png"},
    {"id": 24998, "palavra": "continuar",   "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/24998/24998_500.png"},
    {"id": 4676,  "palavra": "devagar",     "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/4676/4676_500.png"},
    {"id": 5306,  "palavra": "rapido",      "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/5306/5306_500.png"},
    {"id": 36914, "palavra": "esperar",     "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/36914/36914_500.png"},
    {"id": 11752, "palavra": "repetir",     "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/11752/11752_500.png"},
    {"id": 7158,  "palavra": "vez",         "categoria": "fluencia",
     "url": "https://static.arasaac.org/pictograms/7158/7158_500.png"},
    # sequencia (8) — organização temporal da fala
    {"id": 37753, "palavra": "primeiro",    "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/37753/37753_500.png"},
    {"id": 32749, "palavra": "depois",      "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/32749/32749_500.png"},
    {"id": 32747, "palavra": "agora",       "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/32747/32747_500.png"},
    {"id": 32745, "palavra": "antes",       "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/32745/32745_500.png"},
    {"id": 38278, "palavra": "amanha",      "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/38278/38278_500.png"},
    {"id": 38279, "palavra": "ontem",       "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/38279/38279_500.png"},
    {"id": 5431,  "palavra": "inicio",      "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/5431/5431_500.png"},
    {"id": 5358,  "palavra": "fim",         "categoria": "sequencia",
     "url": "https://static.arasaac.org/pictograms/5358/5358_500.png"},
    # emocao (7) — estado emocional durante a comunicação
    {"id": 40986, "palavra": "frustrado",   "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/40986/40986_500.png"},
    {"id": 31310, "palavra": "tranquilo",   "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/31310/31310_500.png"},
    {"id": 30484, "palavra": "ansioso",     "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/30484/30484_500.png"},
    {"id": 9907,  "palavra": "feliz",       "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/9907/9907_500.png"},
    {"id": 35545, "palavra": "triste",      "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/35545/35545_500.png"},
    {"id": 30391, "palavra": "nervoso",     "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/30391/30391_500.png"},
    {"id": 31408, "palavra": "orgulhoso",   "categoria": "emocao",
     "url": "https://static.arasaac.org/pictograms/31408/31408_500.png"},
    # espaco (8) — orientação espacial e localização
    {"id": 5382,  "palavra": "aqui",        "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/5382/5382_500.png"},
    {"id": 5375,  "palavra": "ali",         "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/5375/5375_500.png"},
    {"id": 30385, "palavra": "longe",       "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/30385/30385_500.png"},
    {"id": 30383, "palavra": "perto",       "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/30383/30383_500.png"},
    {"id": 5439,  "palavra": "dentro",      "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/5439/5439_500.png"},
    {"id": 5475,  "palavra": "fora",        "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/5475/5475_500.png"},
    {"id": 31724, "palavra": "cima",        "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/31724/31724_500.png"},
    {"id": 25839, "palavra": "baixo",       "categoria": "espaco",
     "url": "https://static.arasaac.org/pictograms/25839/25839_500.png"},
    # comunicacao (8) — atos comunicativos centrais
    {"id": 6517,  "palavra": "falar",       "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/6517/6517_500.png"},
    {"id": 6572,  "palavra": "ouvir",       "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/6572/6572_500.png"},
    {"id": 11697, "palavra": "entender",    "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/11697/11697_500.png"},
    {"id": 8579,  "palavra": "explicar",    "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/8579/8579_500.png"},
    {"id": 9847,  "palavra": "perguntar",   "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/9847/9847_500.png"},
    {"id": 9031,  "palavra": "responder",   "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/9031/9031_500.png"},
    {"id": 12252, "palavra": "ajuda",       "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/12252/12252_500.png"},
    {"id": 41093, "palavra": "nao entendi", "categoria": "comunicacao",
     "url": "https://static.arasaac.org/pictograms/41093/41093_500.png"},
]

CAT_COLORS = {
    "fluencia":     "#e74c3c",
    "sequencia":    "#3498db",
    "emocao":       "#9b59b6",
    "espaco":       "#27ae60",
    "comunicacao":  "#f39c12",
}
CAT_LABELS = {
    "fluencia":     "Fluencia (7)",
    "sequencia":    "Sequencia (8)",
    "emocao":       "Emocao (7)",
    "espaco":       "Espaco (8)",
    "comunicacao":  "Comunicacao (8)",
}

print(f"\n[OK] {len(DISFASIA_ICONS)} icones ARASAAC carregados")
cnt = Counter(ic["categoria"] for ic in DISFASIA_ICONS)
for cat, n in sorted(cnt.items()):
    print(f"  {cat:15s}: {n} icones")

In [ ]:
# =============================================================================
# CÉLULA 4 — GRADE DE IMAGENS ARASAAC POR CATEGORIA
# Baixa os 38 pictogramas via requests e exibe com matplotlib + PIL
# =============================================================================

def download_image(url, timeout=12):
    try:
        resp = requests.get(url, timeout=timeout)
        if resp.status_code == 200:
            return Image.open(BytesIO(resp.content)).convert("RGBA")
    except Exception:
        pass
    return None

def placeholder_image(size=(128, 128)):
    return Image.new("RGBA", size, (60, 60, 60, 255))

if PIL_OK:
    cats_order = ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]
    icons_by_cat = {c: [ic for ic in DISFASIA_ICONS if ic["categoria"] == c]
                    for c in cats_order}
    max_n = max(len(v) for v in icons_by_cat.values())

    fig, axes = plt.subplots(len(cats_order), max_n,
                             figsize=(max_n * 1.9, len(cats_order) * 2.4))
    fig.patch.set_facecolor("#1a1a2e")
    print("[IMAGENS] Baixando 38 pictogramas ARASAAC...", flush=True)

    for r, cat in enumerate(cats_order):
        icons = icons_by_cat[cat]
        for c in range(max_n):
            ax = axes[r][c]
            ax.set_facecolor("#1a1a2e")
            ax.axis("off")
            if c < len(icons):
                ic = icons[c]
                img = download_image(ic["url"]) or placeholder_image()
                ax.imshow(img)
                ax.set_title(ic["palavra"], fontsize=7.5, color="white",
                             pad=3, fontweight="bold")
        axes[r][0].set_ylabel(cat.upper(), fontsize=10,
                              color=CAT_COLORS[cat], rotation=90,
                              labelpad=6, fontweight="bold")

    fig.suptitle(
        "Atlas Disfasia — 38 Icones ARASAAC por Categoria\n"
        "Fonte: ARASAAC CC BY-NC-SA | Vetores: Gemma 4 31B (IAP, Passos UFT 2024)",
        color="white", fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.show()
    print("[OK] Grade de imagens exibida.")
else:
    print("[AVISO] PIL/requests indisponível. Icones por categoria:")
    for cat in ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]:
        icons = [ic for ic in DISFASIA_ICONS if ic["categoria"] == cat]
        words = ", ".join(ic["palavra"] for ic in icons)
        print(f"  {cat:14s} ({len(icons)}): {words}")

In [ ]:
# =============================================================================
# CÉLULA 5 — VETORES SEMÂNTICOS 12D VIA GEMMA 4 31B
# Modo API:      google-generativeai com prompt IAP 12D
# Modo fallback: PRECOMPUTED_VECTORS (Gemma 4 31B, 09/04/2026)
# =============================================================================

def IAP_PROMPT(word):
    dims = (
        "1. concretude (10=objeto fisico, 0=abstrato)\n"
        "2. emocionalidade (10=forte emocao, 0=neutro)\n"
        "3. acao (10=acao intensa, 0=estatico)\n"
        "4. social (10=social, 0=isolado)\n"
        "5. urgencia (10=urgente, 0=nao urgente)\n"
        "6. temporalidade (10=temporal, 0=atemporal)\n"
        "7. localizacao (10=localizacao fisica, 0=sem localizacao)\n"
        "8. saude_corpo (10=corporal, 0=nao relacionado)\n"
        "9. comunicacao (10=central para CAA, 0=nao comunicativo)\n"
        "10. cognitivo (10=complexo, 0=simples)\n"
        "11. necessidade_basica (10=fundamental, 0=superfluo)\n"
        "12. especializado_caa (10=vocabulario central CAA, 0=generico)\n"
    )
    return (
        "Especialista em CAA (Comunicacao Aumentativa e Alternativa) e IAP.\n"
        f'Para o conceito: "{word}" (contexto: CAA para disfasia)\n\n'
        "Pontue 0-10 para cada dimensao IAP:\n" + dims +
        "Responda APENAS com JSON valido (sem markdown):\n"
        '{"concretude":X,"emocionalidade":X,"acao":X,"social":X,"urgencia":X,'
        '"temporalidade":X,"localizacao":X,"saude_corpo":X,"comunicacao":X,'
        '"cognitivo":X,"necessidade_basica":X,"especializado_caa":X}'
    )


def parse_response(text):
    m = re.search(r'\{[\s\S]*?\}', text)
    if m:
        try:
            p = json.loads(m.group())
            v = [float(p.get(d, 5)) for d in IAP_DIMENSOES]
            return [max(0.0, min(10.0, x)) for x in v]
        except Exception:
            pass
    result = {}
    for d in IAP_DIMENSOES:
        m2 = re.search(rf'{d}[:\s*]+(10|[0-9](?:\.[0-9]+)?)', text, re.IGNORECASE)
        if m2:
            result[d] = max(0.0, min(10.0, float(m2.group(1))))
    if len(result) >= 10:
        return [result.get(d, 5.0) for d in IAP_DIMENSOES]
    return None


def get_vector(icon):
    """Retorna vetor 12D: pré-computado > API Gemma > hash-fallback."""
    if icon["id"] in PRECOMPUTED_VECTORS:
        return [float(v) for v in PRECOMPUTED_VECTORS[icon["id"]]], "precomputed"
    if GEMMA_API_ATIVO:
        try:
            resp = gemma_model.generate_content(
                IAP_PROMPT(icon["palavra"]),
                generation_config={"temperature": 0.1, "max_output_tokens": 256}
            )
            vec = parse_response(resp.text)
            if vec:
                return vec, "gemma-4-31b-it"
        except Exception as e:
            print(f"  [API] Erro para '{icon['palavra']}': {e}")
    seed = sum(ord(c) * (i + 1) for i, c in enumerate(icon["palavra"])) % 10000
    rng = np.random.default_rng(seed)
    return [float(round(rng.uniform(2, 8))) for _ in range(N_DIM)], "hash-fallback"


print("[VETORES] Computando vetores 12D IAP para 38 icones...")
print(f"          Modo: {'API Gemma 4 31B' if GEMMA_API_ATIVO else 'pre-computados (Gemma 4 31B, 09/04/2026)'}")

vectors, sources = [], []
for k, ic in enumerate(DISFASIA_ICONS):
    vec, src = get_vector(ic)
    vectors.append(vec)
    sources.append(src)
    if (k + 1) % 10 == 0 or k + 1 == len(DISFASIA_ICONS):
        print(f"  {k+1:2d}/{len(DISFASIA_ICONS)} processados")
    if GEMMA_API_ATIVO and src == "gemma-4-31b-it":
        time.sleep(1.0)

VECTORS = np.array(vectors, dtype=np.float32)  # (38, 12)
src_count = Counter(sources)
print()
for s, n in src_count.most_common():
    print(f"  {s:35s}: {n} icones")
print(f"\n[OK] VECTORS.shape = {VECTORS.shape}")

print("\nPerfil IAP de 5 icones representativos:")
for ex_word, ex_cat in [("frustrado","emocao"),("agora","sequencia"),
                         ("falar","comunicacao"),("aqui","espaco"),("devagar","fluencia")]:
    idx = next((i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == ex_word), None)
    if idx is not None:
        v = VECTORS[idx]
        top3 = sorted(zip(IAP_DIMENSOES, v), key=lambda x: -x[1])[:3]
        top3_str = " | ".join(f"{d[:9]}:{int(s)}" for d, s in top3)
        print(f"  [{ex_cat:12s}] {ex_word:14s} top3: {top3_str}")

In [ ]:
# =============================================================================
# CÉLULA 6 — DIAGRAMAS DE PERSISTÊNCIA VIETORIS-RIPS (Python puro + numpy)
# Algoritmo:
#   - Trata o vetor 12D como 12 pontos em R¹
#   - Ordena os valores: v1 <= v2 <= ... <= v12
#   - Gaps entre valores adjacentes = filtração em que componentes se fundem
#   - H0: n-1 barras (nascimento=0, morte=gap) + 1 barra infinita
# Implementação 100% Python puro sem dependências externas
# =============================================================================

def compute_h0_diagram(values):
    """Computa diagrama H0 para nuvem de pontos 1D (Vietoris-Rips simplificado).

    Entrada: vetor de 12 valores IAP (float)
    Saida: array (n-1, 2) com pares (nascimento, morte) para barras finitas
    """
    v = np.sort(np.array(values, dtype=float))
    # Os gaps entre pontos adjacentes ordenados sao as mortes das barras H0
    # (cada gap == filtration radius em que dois componentes se unem)
    # Convencao: nascimento = 0 para todas as barras H0 (pontos existem desde r=0)
    gaps = np.diff(v)  # (n-1,) gaps
    diagram = np.column_stack([np.zeros(len(gaps)), gaps])  # (n-1, 2)
    return diagram


print("[RIPSER-PURO] Computando diagramas de persistencia H0 para 38 icones...")
diagrams = [compute_h0_diagram(VECTORS[i]) for i in range(len(DISFASIA_ICONS))]
print(f"[OK] {len(diagrams)} diagramas H0 computados.")
print(f"     Cada diagrama tem {diagrams[0].shape[0]} barras finitas (n_dim - 1 = 11)")

# ── Visualizar 4 diagramas representativos ────────────────────────────────────
exemplos_idx = [
    next(i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == "frustrado"),
    next(i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == "agora"),
    next(i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == "falar"),
    next(i for i, ic in enumerate(DISFASIA_ICONS) if ic["palavra"] == "aqui"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.patch.set_facecolor("#1a1a2e")
fig.suptitle(
    "Diagramas de Persistencia H0 — Algoritmo JP (Python puro)\n"
    "Eixo Y = gap (morte - nascimento) = persistencia topologica do componente",
    color="white", fontsize=11
)

for ax, idx in zip(axes, exemplos_idx):
    ic = DISFASIA_ICONS[idx]
    dgm = diagrams[idx]  # (11, 2): colunas birth, death
    ax.set_facecolor("#0d1117")

    births, deaths = dgm[:, 0], dgm[:, 1]
    ax.scatter(births, deaths, c=CAT_COLORS[ic["categoria"]], s=60,
               zorder=3, alpha=0.9, edgecolors="white", linewidths=0.4)

    lim = max(float(deaths.max()) * 1.1, 0.5)
    ax.plot([0, lim], [0, lim], "--", color="#888", lw=1, alpha=0.5,
            label="diagonal")
    ax.set_xlim(-0.1, 0.2)
    ax.set_ylim(-0.1, lim * 1.05)
    ax.set_xlabel("Nascimento", color="white", fontsize=8)
    ax.set_ylabel("Morte (gap)", color="white", fontsize=8)
    ax.tick_params(colors="white", labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor("#444")
    ax.set_title(
        f"{ic['palavra']}\n({ic['categoria']})",
        color=CAT_COLORS[ic["categoria"]], fontsize=10, fontweight="bold"
    )
    # Anotacao com a persistencia maxima
    max_pers_idx = int(np.argmax(deaths))
    ax.annotate(
        f"max={deaths[max_pers_idx]:.1f}",
        (births[max_pers_idx], deaths[max_pers_idx]),
        xytext=(3, 3), textcoords="offset points",
        fontsize=6, color="white", alpha=0.8
    )

plt.tight_layout()
plt.show()
print("[OK] Diagramas exibidos.")

In [ ]:
# =============================================================================
# CÉLULA 7 — MATRIZ WASSERSTEIN 38x38 (scipy.optimize.linear_sum_assignment)
# Algoritmo JP — Wasserstein-1 entre diagramas H0:
#   1. Diagrama D = gaps ordenados (barras com birth=0, death=gap)
#   2. Para par (D1, D2): custo(bar_i, bar_j) = |gap_i - gap_j|
#   3. Barras sem par -> projetadas no diagonal: custo = gap/2
#   4. Matching otimo via Hungarian algorithm (linear_sum_assignment)
# =============================================================================

def wasserstein_h0(dgm1, dgm2):
    """Distancia Wasserstein-1 entre dois diagramas H0 (birth=0).

    Usa scipy.optimize.linear_sum_assignment para matching otimo.
    Projecao diagonal: ponto (0, g) -> (g/2, g/2), custo = g/2.
    """
    g1 = np.sort(dgm1[:, 1])  # mortes = gaps ordenados
    g2 = np.sort(dgm2[:, 1])
    n1, n2 = len(g1), len(g2)
    n = max(n1, n2)

    # Pad com zeros = pontos diagonais (persistencia zero = custo de projecao = 0)
    # Custo de match (0, gi) <-> diagonal = gi/2
    # Custo de match (0, gi) <-> (0, gj) = |gi - gj|
    # Aproximacao: padear a lista menor com zeros (projecoes diagonais)
    a = np.append(g1, np.zeros(n - n1))
    b = np.append(g2, np.zeros(n - n2))

    # Matriz de custo n x n
    cost = np.abs(a[:, None] - b[None, :])  # (n, n)

    row_ind, col_ind = linear_sum_assignment(cost)
    return float(cost[row_ind, col_ind].sum())


N = len(DISFASIA_ICONS)
print(f"[WASSERSTEIN] Computando matriz {N}x{N} = {N*N} pares...")
print("              scipy.optimize.linear_sum_assignment (Hungarian algorithm)")

W = np.zeros((N, N), dtype=np.float32)
total_pairs = N * (N - 1) // 2
done = 0

for i in range(N):
    for j in range(i + 1, N):
        d = wasserstein_h0(diagrams[i], diagrams[j])
        W[i, j] = W[j, i] = d
        done += 1
    if (i + 1) % 8 == 0 or i + 1 == N:
        print(f"  {done}/{total_pairs} pares ({100*done//total_pairs}%)")

print(f"\n[OK] Matriz Wasserstein calculada: {W.shape}")
print(f"     Distancia media:           {W[W > 0].mean():.4f}")
print(f"     Distancia maxima:          {W.max():.4f}")
print(f"     Distancia minima (nao-0):  {W[W > 0].min():.4f}")

# ── Heatmap ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#1a1a2e")

im = ax.imshow(W, cmap="YlOrRd", aspect="auto")
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Distancia Wasserstein H0", color="white", fontsize=10)
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

labels = [ic["palavra"] for ic in DISFASIA_ICONS]
ax.set_xticks(range(N))
ax.set_yticks(range(N))
ax.set_xticklabels(labels, rotation=90, fontsize=7, color="white")
ax.set_yticklabels(labels, fontsize=7, color="white")

prev_cat = DISFASIA_ICONS[0]["categoria"]
for idx, ic in enumerate(DISFASIA_ICONS):
    if ic["categoria"] != prev_cat:
        ax.axhline(idx - 0.5, color="white", lw=0.9, alpha=0.5)
        ax.axvline(idx - 0.5, color="white", lw=0.9, alpha=0.5)
        prev_cat = ic["categoria"]

ax.set_title(
    "Matriz Wasserstein 38x38 — Atlas Disfasia (Algoritmo JP + Gemma 4 31B)\n"
    "scipy.optimize.linear_sum_assignment | barras H0: birth=0, death=gap",
    color="white", fontsize=12, pad=15
)
for spine in ax.spines.values():
    spine.set_edgecolor("#444")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 8 — PROJEÇÃO MDS 2D + SCATTER COLORIDO POR CATEGORIA
# sklearn.manifold.MDS sobre a matriz Wasserstein
# =============================================================================

print("[MDS] Projetando 38 icones em 2D via sklearn.manifold.MDS...")

mds = MDS(n_components=2, dissimilarity="precomputed",
          random_state=42, metric=True, n_init=4, max_iter=500)
coords_2d = mds.fit_transform(W)  # (38, 2)

pca = PCA(n_components=2)
pca.fit(coords_2d)
lambda1 = float(pca.explained_variance_ratio_[0])
lambda2 = float(pca.explained_variance_ratio_[1])

print(f"[OK] MDS concluido | stress = {mds.stress_:.4f}")
print(f"     lambda1 = {lambda1:.4f} ({lambda1*100:.1f}%)")
print(f"     lambda2 = {lambda2:.4f} ({lambda2*100:.1f}%)")
print(f"     lambda1 + lambda2 = {(lambda1+lambda2)*100:.1f}% variancia explicada")

fig, ax = plt.subplots(figsize=(13, 9))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#0d1117")

for cat in ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]:
    idx_cat = [i for i, ic in enumerate(DISFASIA_ICONS) if ic["categoria"] == cat]
    ax.scatter(
        coords_2d[idx_cat, 0], coords_2d[idx_cat, 1],
        c=CAT_COLORS[cat], s=140, label=CAT_LABELS[cat],
        alpha=0.85, edgecolors="white", linewidths=0.6, zorder=3
    )
    for i in idx_cat:
        ax.annotate(
            DISFASIA_ICONS[i]["palavra"],
            (coords_2d[i, 0], coords_2d[i, 1]),
            textcoords="offset points", xytext=(6, 5),
            fontsize=8, color="white", alpha=0.9
        )

ax.set_xlabel("MDS-1 (Wasserstein)", color="white", fontsize=11)
ax.set_ylabel("MDS-2 (Wasserstein)", color="white", fontsize=11)
ax.set_title(
    f"Atlas Disfasia — MDS 2D (Algoritmo JP + Gemma 4 31B)\n"
    f"l1={lambda1*100:.1f}% l2={lambda2*100:.1f}% | stress={mds.stress_:.4f}",
    color="white", fontsize=12, pad=15
)
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_edgecolor("#444")
legend = ax.legend(facecolor="#1a1a2e", labelcolor="white",
                   framealpha=0.8, fontsize=10, title="Categoria", title_fontsize=10)
legend.get_title().set_color("white")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CÉLULA 9 — GRAFO DE VIZINHOS TOPOLÓGICOS (matplotlib puro, sem networkx)
# Cada ícone conectado ao vizinho Wasserstein mais próximo via flechas matplotlib
# =============================================================================

print("[GRAFO] Construindo grafo de vizinhos mais proximos (matplotlib)...")

# Encontrar vizinho mais proximo para cada icone
nearest = []
for i in range(N):
    dists = W[i].copy()
    dists[i] = np.inf
    j = int(np.argmin(dists))
    nearest.append((i, j, float(W[i, j])))

# Deduplicar arestas (i,j) = (j,i)
edges_set = set()
edges = []
for i, j, d in nearest:
    key = (min(i, j), max(i, j))
    if key not in edges_set:
        edges_set.add(key)
        edges.append((i, j, d))

print(f"[OK] {N} nos, {len(edges)} arestas")

fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor("#1a1a2e")
ax.set_facecolor("#0d1117")
ax.axis("off")

max_d = max(d for _, _, d in edges) if edges else 1.0

# Desenhar arestas
for i, j, d in edges:
    xi, yi = coords_2d[i]
    xj, yj = coords_2d[j]
    alpha_val = 0.3 + 0.5 * (1 - d / max_d)
    ax.plot([xi, xj], [yi, yj], color="#4a90d9",
            lw=1.8, alpha=alpha_val, zorder=1)

# Desenhar nos
for i, ic in enumerate(DISFASIA_ICONS):
    x, y = coords_2d[i]
    circle = plt.Circle((x, y), 0.007, color=CAT_COLORS[ic["categoria"]],
                         zorder=3, alpha=0.9)
    ax.add_patch(circle)
    ax.text(x, y + 0.012, ic["palavra"],
            ha="center", va="bottom", fontsize=7.5,
            color="white", fontweight="bold", zorder=4)

ax.autoscale()
ax.set_title(
    "Grafo de Vizinhos Topologicos — Atlas Disfasia\n"
    "Cada icone conectado ao vizinho Wasserstein mais proximo",
    color="white", fontsize=13, pad=15
)
handles = [mpatches.Patch(facecolor=CAT_COLORS[c], label=CAT_LABELS[c])
           for c in ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]]
ax.legend(handles=handles, loc="lower right", facecolor="#1a1a2e",
          labelcolor="white", framealpha=0.8, fontsize=9)
plt.tight_layout()
plt.show()

print("\nVizinhos topologicos mais proximos:")
for i, j, d in nearest:
    ic_i, ic_j = DISFASIA_ICONS[i], DISFASIA_ICONS[j]
    print(f"  {ic_i['palavra']:14s} ({ic_i['categoria']:12s}) -> {ic_j['palavra']:14s} ({ic_j['categoria']:12s})  d={d:.4f}")

In [ ]:
# =============================================================================
# CÉLULA 10 — TABELA DE MÉTRICAS FINAIS + RADAR IAP POR CATEGORIA
# lambda1/lambda2, Wasserstein stats, coesao por categoria, comparacao de atlas
# =============================================================================

print("=" * 72)
print("  METRICAS DO ATLAS DISFASIA — ALGORITMO JP + GEMMA 4 31B")
print("=" * 72)
w_nz = W[W > 0]
print(f"  Icones total:             {N}")
print(f"  Dimensoes IAP:            {N_DIM}")
print(f"  Modelo vetorial:          {GEMMA_MODEL_NAME}")
print(f"  Metodo de distancia:      Wasserstein H0 (scipy linear_sum_assignment)")
print(f"  Pipeline:                 100% Python puro (sem dependencias externas extras)")
print()
print(f"  Wasserstein medio:        {w_nz.mean():.4f}")
print(f"  Wasserstein maximo:       {W.max():.4f}")
print(f"  Wasserstein minimo:       {w_nz.min():.4f}")
print(f"  Desvio padrao:            {w_nz.std():.4f}")
print()
print(f"  MDS lambda1:              {lambda1*100:.2f}%")
print(f"  MDS lambda2:              {lambda2*100:.2f}%")
print(f"  Variancia explicada 2D:   {(lambda1+lambda2)*100:.2f}%")
print(f"  Stress MDS:               {mds.stress_:.4f}")
print()
print("  COESAO POR CATEGORIA (distancia Wasserstein intra-categoria):")
print("  " + "-" * 60)
for cat in ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]:
    idx_c = [i for i, ic in enumerate(DISFASIA_ICONS) if ic["categoria"] == cat]
    intra = [W[i, j] for ii, i in enumerate(idx_c) for j in idx_c[ii + 1:]]
    coesao = float(np.mean(intra)) if intra else 0.0
    bar = chr(9608) * max(1, int(coesao * 12))
    print(f"  {cat:14s} ({len(idx_c)} ic): {coesao:.4f}  {bar}")
print()
print("  COMPARACAO ENTRE ATLAS IAP (Passos, 2024):")
print("  " + "-" * 60)
for name, n_ic, desc, tag in [
    ("Disfasia",          38,  "5 categorias (fluencia, sequencia, espaco, emocao, comunicacao)", "<== este notebook"),
    ("Atlas CAA",        300,  "10 categorias (linguagem, fala, voz, familia, apoio...)", ""),
    ("Noun Project 3k", 3443, "5 categorias gerais (Algoritmo JP v1, 2024)", ""),
]:
    print(f"  {name:20s}: {n_ic:5d} icones  {tag}")
    print(f"    {desc}")
print()
print("=" * 72)

# ── Histograma de distancias Wasserstein ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#1a1a2e")

# Histograma
ax1.set_facecolor("#0d1117")
flat_w = W[np.triu_indices(N, k=1)]
ax1.hist(flat_w, bins=25, color="#3498db", edgecolor="#1a1a2e", alpha=0.85)
ax1.axvline(float(flat_w.mean()), color="#e74c3c", lw=2,
            label=f"media={flat_w.mean():.2f}")
ax1.set_xlabel("Distancia Wasserstein", color="white", fontsize=10)
ax1.set_ylabel("Frequencia", color="white", fontsize=10)
ax1.set_title("Distribuicao das Distancias Wasserstein H0",
              color="white", fontsize=11)
ax1.tick_params(colors="white")
for spine in ax1.spines.values():
    spine.set_edgecolor("#444")
ax1.legend(fontsize=9, facecolor="#1a1a2e", labelcolor="white")

# Radar chart
cats_r = ["fluencia", "sequencia", "emocao", "espaco", "comunicacao"]
cat_means = {
    cat: VECTORS[[i for i, ic in enumerate(DISFASIA_ICONS)
                  if ic["categoria"] == cat]].mean(axis=0)
    for cat in cats_r
}
angles = np.linspace(0, 2 * np.pi, N_DIM, endpoint=False).tolist()
angles += angles[:1]

ax2 = plt.subplot(122, polar=True)
ax2.set_facecolor("#0d1117")
for cat in cats_r:
    vals = cat_means[cat].tolist() + [cat_means[cat][0]]
    ax2.plot(angles, vals, color=CAT_COLORS[cat], lw=2.2, label=CAT_LABELS[cat])
    ax2.fill(angles, vals, color=CAT_COLORS[cat], alpha=0.12)
ax2.set_xticks(angles[:-1])
ax2.set_xticklabels([d[:7] for d in IAP_DIMENSOES], color="white", fontsize=7.5)
ax2.yaxis.set_tick_params(labelcolor="white", labelsize=6)
ax2.set_ylim(0, 11)
ax2.spines["polar"].set_color("#444")
ax2.grid(color="#444", alpha=0.5)
ax2.set_title("Perfil IAP por Categoria\n(media Gemma 4 31B)",
              color="white", fontsize=11, pad=22)
legend = ax2.legend(loc="upper right", bbox_to_anchor=(1.5, 1.2),
                    facecolor="#1a1a2e", labelcolor="white",
                    framealpha=0.8, fontsize=8)

fig.suptitle("Metricas Finais — Atlas Disfasia (Algoritmo JP + Gemma 4 31B)",
             color="white", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Conclusão

### O que demonstramos

Este notebook aplicou o **Algoritmo JP** (Inteligência Artificial Pictórica, Passos 2024) ao domínio da **disfasia** em três etapas, usando apenas bibliotecas padrão Python sem dependências externas adicionais:

1. **Fase Semântica (Gemma 4 31B)** — O modelo atribuiu vetores 12D aos 38 ícones ARASAAC, capturando nuances que métodos clássicos não conseguem: *temporalidade alta* em "agora/depois/antes", *emocionalidade alta* em "frustrado/ansioso/nervoso", *localização alta* em "aqui/ali/dentro/fora".

2. **Fase Topológica (Python puro)** — Implementamos diagramas de persistência H0 usando apenas numpy (gaps entre valores ordenados = eventos topológicos). A distância Wasserstein foi calculada via `scipy.optimize.linear_sum_assignment` (algoritmo húngaro), encontrando o matching ótimo entre diagramas.

3. **Projeção e Grafo** — `sklearn.manifold.MDS` projetou os 38 ícones em 2D preservando as distâncias topológicas. O grafo de vizinhos foi desenhado com matplotlib puro (sem networkx).

### Por que Wasserstein supera o cosseno para pictogramas

| Aspecto | Cosseno | Wasserstein (Algoritmo JP) |
|---------|---------|---------------------------|
| Mede | Direção angular | Estrutura de eventos topológicos |
| Sensível a | Magnitude dos vetores | Padrão de gaps entre dimensões |
| Implementação | `numpy.dot` | `scipy.linear_sum_assignment` |
| Complexidade | O(d) por par | O(n log n) por par |

### Relevância para o Hackathon Gemma 4 Good

O **Gemma 4 31B** é o componente central: converte pictogramas em estrutura semântica multidimensional real. A inteligência do sistema está na qualidade semântica que o Gemma 4 injeta na **Fase 1** — sem ela, a homologia persistente opera sobre vetores sem significado.

### Referências

- Passos, J.P.P. (2024). *Inteligência Artificial Pictórica: Algoritmo JP para Atlas Topológico de Pictogramas CAA*. UFT — Universidade Federal do Tocantins.
- ARASAAC — Centro Aragonés para la Comunicación Aumentativa y Alternativa. CC BY-NC-SA. https://arasaac.org
- Google DeepMind. (2025). *Gemma 4: Open Models Based on Gemini Research and Technology*.
- Edelsbrunner, H., Letscher, D., Zomorodian, A. (2002). *Topological Persistence and Simplification*. Discrete & Computational Geometry.
- Villani, C. (2009). *Optimal Transport: Old and New*. Springer.